# Taller resuelto: K-Means en imágenes

Este notebook sigue la misma estructura pedagógica:
1. Antes del bloque: qué se va a hacer.
2. Dentro del bloque: comentarios línea por línea.
3. Después del bloque: interpretación.

## Objetivo
Segmentar una imagen usando K-Means y comparar RGB vs HSV.


## Bloque 1. Importar librerías y cargar una imagen

En este bloque vamos a cargar una imagen real, convertirla a RGB y HSV, y visualizar ambas versiones.


In [ ]:
# Importamos NumPy.
import numpy as np

# Importamos Matplotlib.
import matplotlib.pyplot as plt

# Importamos KMeans.
from sklearn.cluster import KMeans

# Importamos una imagen de ejemplo.
from sklearn.datasets import load_sample_image

# Importamos conversión RGB a HSV.
from matplotlib.colors import rgb_to_hsv

# Cargamos la imagen china.jpg.
original_image = load_sample_image("china.jpg")

# Reducimos el tamaño para hacer el cálculo más rápido.
original_image = original_image[::3, ::3]

# Normalizamos la imagen entre 0 y 1.
img_rgb = original_image / 255.0

# Convertimos RGB a HSV.
img_hsv = rgb_to_hsv(img_rgb)

# Creamos figura.
_, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mostramos RGB.
axes[0].imshow(img_rgb)
axes[0].set_axis_off()
axes[0].set_title("Imagen RGB")

# Mostramos HSV.
axes[1].imshow(img_hsv)
axes[1].set_axis_off()
axes[1].set_title("Imagen HSV")

# Mostramos la figura.
plt.show()


### Interpretación del bloque 1

La misma imagen puede representarse en distintos espacios de color.

**Qué significa:**
- RGB mezcla información de color y brillo.
- HSV separa tono, saturación y valor.
- Esto puede cambiar el comportamiento del clustering.


## Bloque 2. Vectorizar la imagen

En este bloque vamos a convertir la imagen en una matriz donde cada fila sea un píxel y cada columna un canal de color.


In [ ]:
# Reorganizamos la imagen RGB en una matriz de píxeles.
vectorized_rgb = img_rgb.reshape((-1, 3))

# Convertimos a float32.
vectorized_rgb = np.float32(vectorized_rgb)

# Reorganizamos la imagen HSV en una matriz de píxeles.
vectorized_hsv = img_hsv.reshape((-1, 3))

# Convertimos a float32.
vectorized_hsv = np.float32(vectorized_hsv)

# Mostramos dimensiones.
print(vectorized_rgb.shape)
print(vectorized_hsv.shape)


### Interpretación del bloque 2

Ahora la imagen se comporta como un dataset tabular.

**Idea clave:**
- cada píxel es una observación
- cada canal de color es una variable
- K-Means agrupará píxeles similares


## Bloque 3. Segmentar en RGB

En este bloque entrenamos K-Means sobre los píxeles RGB y reconstruimos la segmentación.


In [ ]:
# Definimos el número de clusters.
n = 5

# Entrenamos K-Means en RGB.
kmeans = KMeans(n_clusters=n, random_state=0).fit(vectorized_rgb)

# Predecimos el cluster de cada píxel.
clustered_rgb = kmeans.predict(vectorized_rgb)

# Reorganizamos el resultado con la forma de la imagen.
clustered_rgb = clustered_rgb.reshape(img_rgb.shape[:2])

# Mostramos dimensiones.
print(vectorized_rgb.shape, img_rgb.shape, clustered_rgb.shape)


### Interpretación del bloque 3

El algoritmo ya asignó cada píxel a un grupo de color.

**Qué significa:**
- todavía no estamos reconociendo objetos
- sí estamos descubriendo regiones con similitud cromática


## Bloque 4. Visualizar segmentación RGB y aislar un cluster

En este bloque mostramos la segmentación y generamos una máscara para un cluster específico.


In [ ]:
# Creamos figura comparativa.
_, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mostramos imagen original.
axes[0].imshow(img_rgb)
axes[0].set_axis_off()
axes[0].set_title("Original")

# Mostramos segmentación RGB.
axes[1].imshow(clustered_rgb, cmap="Paired")
axes[1].set_axis_off()
axes[1].set_title("Segmentación RGB")

# Mostramos la figura.
plt.show()

# Elegimos un cluster.
cluster = 0

# Creamos máscara binaria.
cluster_mask = np.array(clustered_rgb == cluster, dtype=np.uint8)

# Copiamos la imagen.
masked_image = np.copy(img_rgb)

# Ponemos en negro los píxeles que no pertenecen al cluster.
masked_image[cluster_mask == 0] = 0

# Mostramos la máscara.
plt.figure(figsize=(6, 5))
plt.imshow(masked_image)
plt.axis("off")
plt.title("Cluster aislado en RGB")
plt.show()


### Interpretación del bloque 4

La máscara permite ver con claridad qué región de la imagen pertenece al cluster elegido.

**Idea clave:**
Segmentar no es reconocer semánticamente, sino separar regiones por similitud.


## Bloque 5. Segmentar en HSV

En este bloque repetimos el proceso en HSV para comparar cómo cambia la segmentación.


In [ ]:
# Definimos número de clusters.
n = 5

# Entrenamos K-Means en HSV.
kmeans = KMeans(n_clusters=n, random_state=0).fit(vectorized_hsv)

# Predecimos clusters.
clustered_hsv = kmeans.predict(vectorized_hsv)

# Reorganizamos con la forma de la imagen.
clustered_hsv = clustered_hsv.reshape(img_rgb.shape[:2])

# Mostramos dimensiones.
print(vectorized_hsv.shape, img_rgb.shape, clustered_hsv.shape)


### Interpretación del bloque 5

Aquí el algoritmo trabaja con una representación distinta del color.

**Qué cambia:**
- HSV puede ser más robusto frente a cambios de iluminación.
- Eso puede producir segmentaciones más coherentes en ciertas imágenes.


## Bloque 6. Visualizar segmentación HSV y comparar

En este bloque mostramos la segmentación HSV y una máscara para compararla con RGB.


In [ ]:
# Creamos figura comparativa.
_, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mostramos imagen original.
axes[0].imshow(img_rgb)
axes[0].set_axis_off()
axes[0].set_title("Original")

# Mostramos segmentación HSV.
axes[1].imshow(clustered_hsv, cmap="Paired")
axes[1].set_axis_off()
axes[1].set_title("Segmentación HSV")

# Mostramos la figura.
plt.show()

# Elegimos un cluster.
cluster = 0

# Creamos máscara binaria.
cluster_mask = np.array(clustered_hsv == cluster, dtype=np.uint8)

# Copiamos imagen original.
masked_image = np.copy(img_rgb)

# Ponemos en negro lo que no pertenece al cluster.
masked_image[cluster_mask == 0] = 0

# Mostramos máscara.
plt.figure(figsize=(6, 5))
plt.imshow(masked_image)
plt.axis("off")
plt.title("Cluster aislado en HSV")
plt.show()


### Interpretación del bloque 6

Comparar RGB y HSV permite discutir qué representación del color conviene más.

**Mensaje docente:**
No siempre el problema está en el algoritmo; muchas veces está en cómo representamos los datos.


## Cierre del ejercicio

### Ideas clave
- una imagen es un conjunto de píxeles
- K-Means puede segmentar regiones por color
- RGB y HSV pueden producir resultados diferentes
- elegir la representación correcta es parte del análisis
